<a href="https://colab.research.google.com/github/mohammedAlkhuzaie/Suha-Ali-Salman/blob/main/Ducoment_editor_Suha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
Document Editor
================

Problem
-------
A document editor must support multiple file formats (PDF, Word, HTML, and
any future format) without the editor's core logic being coupled to any
concrete format implementation, and without needing to modify the editor
whenever a new format is added.

Design
------
This uses the **Factory Method** pattern combined with a **format registry**:

- `Document` is an abstract base class defining the contract every concrete
  format must satisfy (`save`, `render`).
- Each concrete format (`PDFDocument`, `WordDocument`, `HTMLDocument`, ...)
  implements that contract independently.
- `DocumentFactory` is a registry-based factory. New formats register
  themselves via `DocumentFactory.register(name, creator)` instead of the
  factory containing an if/elif chain over known types. This satisfies the
  Open/Closed Principle: the factory and the `DocumentEditor` are closed for
  modification but open for extension.
- `DocumentEditor` (the core application logic) depends only on the
  `Document` abstraction and on `DocumentFactory`. It never imports or
  references a concrete format class, so adding a new format never requires
  touching the editor.
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Callable, Dict, List


class Document(ABC):
    """Abstract contract that every document format must implement."""

    def __init__(self, content: str = "") -> None:
        self.content = content

    @abstractmethod
    def save(self, path: str) -> str:
        """Persist the document and return the path/identifier it was saved to."""
        raise NotImplementedError

    @abstractmethod
    def render(self) -> str:
        """Return a human-readable representation of the document."""
        raise NotImplementedError

    @property
    @abstractmethod
    def extension(self) -> str:
        raise NotImplementedError


class PDFDocument(Document):
    def save(self, path: str) -> str:
        full_path = f"{path}{self.extension}"
        # In a real implementation this would invoke a PDF-writing library.
        return full_path

    def render(self) -> str:
        return f"[PDF] {self.content}"

    @property
    def extension(self) -> str:
        return ".pdf"


class WordDocument(Document):
    def save(self, path: str) -> str:
        full_path = f"{path}{self.extension}"
        # In a real implementation this would invoke python-docx or similar.
        return full_path

    def render(self) -> str:
        return f"[WORD] {self.content}"

    @property
    def extension(self) -> str:
        return ".docx"


class HTMLDocument(Document):
    def save(self, path: str) -> str:
        full_path = f"{path}{self.extension}"
        return full_path

    def render(self) -> str:
        return f"<html><body>{self.content}</body></html>"

    @property
    def extension(self) -> str:
        return ".html"


class UnsupportedFormatError(ValueError):
    """Raised when the editor is asked for a format that has not been registered."""


class DocumentFactory:
    """Registry-based factory. Formats register themselves; the factory
    never needs an if/elif chain, and the editor never needs to change."""

    _registry: Dict[str, Callable[[str], Document]] = {}

    @classmethod
    def register(cls, format_name: str, creator: Callable[[str], Document]) -> None:
        cls._registry[format_name.lower()] = creator

    @classmethod
    def create(cls, format_name: str, content: str = "") -> Document:
        key = format_name.lower()
        if key not in cls._registry:
            raise UnsupportedFormatError(
                f"No document type registered for format '{format_name}'. "
                f"Available: {sorted(cls._registry)}"
            )
        return cls._registry[key](content)

    @classmethod
    def available_formats(cls) -> List[str]:
        return sorted(cls._registry)


# Built-in formats register themselves at import time. Adding a brand-new
# format later (e.g. Markdown) only requires one call to `register` in that
# format's own module -- nothing here or in DocumentEditor changes.
DocumentFactory.register("pdf", PDFDocument)
DocumentFactory.register("word", WordDocument)
DocumentFactory.register("html", HTMLDocument)


class DocumentEditor:
    """Core editor logic. Depends only on the Document abstraction and the
    factory -- never on a concrete format class."""

    def __init__(self) -> None:
        self._document: Document | None = None

    def new_document(self, format_name: str, content: str = "") -> Document:
        self._document = DocumentFactory.create(format_name, content)
        return self._document

    def edit(self, content: str) -> None:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        self._document.content = content

    def display(self) -> str:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        return self._document.render()

    def save(self, path: str) -> str:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        return self._document.save(path)

    @property
    def current_document(self) -> Document | None:
        return self._document


if __name__ == "__main__":
    editor = DocumentEditor()
    editor.new_document("pdf", "Quarterly report draft")
    print(editor.display())
    print(editor.save("/tmp/report"))
    print("Available formats:", DocumentFactory.available_formats())


[PDF] Quarterly report draft
/tmp/report.pdf
Available formats: ['html', 'pdf', 'word']
